In [1]:
import numpy as np
import polars as pl
from pathlib import Path

from src.models.data import fp_from_smiles
from src.models.pipeline import results_table, run_parallel_trainings, train_and_score, tune_gnn
from src.models import config as model_cfg
from src._config import PROCESSED_DATA

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(100)

/home/computer/Repositories/ml_chembl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


polars.config.Config

In [2]:
df = pl.read_parquet(PROCESSED_DATA / "ChEMBL_processed.parquet")

In [3]:
df_temp = df.with_columns(
    pl.col("canonical_smiles").map_elements(fp_from_smiles, return_dtype=pl.Object).alias("fp")
)
df_clean = df_temp.filter(
    (pl.col("fp").is_not_null()) & 
    (pl.col("pIC50").is_not_null()) &
    (pl.col("pIC50").is_not_nan())
)
print(f"Liczba probek po czyszczeniu: {len(df_clean)}")
print(f"Odrzucono {len(df) - len(df_clean)} blednych czasteczek.")

Liczba probek po czyszczeniu: 15723
Odrzucono 0 blednych czasteczek.


In [4]:
from src.models.config import (
    EPOCHS_DEFAULT, LR_DEFAULT, BATCH_SIZE_DEFAULT, SEED_DEFAULT,
    EARLY_STOPPING_PATIENCE_DEFAULT, MIN_DELTA_DEFAULT, WEIGHT_DECAY_DEFAULT, POOLING_DEFAULT,
)

print(f"Domyslne parametry: epochs={EPOCHS_DEFAULT}, lr={LR_DEFAULT}, batch={BATCH_SIZE_DEFAULT}, "
      f"seed={SEED_DEFAULT}, patience={EARLY_STOPPING_PATIENCE_DEFAULT}, pooling={POOLING_DEFAULT}")

fp_has_nan = any(np.isnan(fp).any() for fp in df_clean["fp"])
print(f"NaN w fingerprintach: {fp_has_nan}")

Domyslne parametry: epochs=200, lr=0.0003, batch=64, seed=42, patience=15, pooling=mean
NaN w fingerprintach: False


In [5]:
MLP_SPECS = [
    {"model_type": "MLP", "split_type": "random",   "df_fp": df_clean, "log_mlflow": True, "evaluate_test": False},
    {"model_type": "MLP", "split_type": "scaffold", "df_fp": df_clean, "log_mlflow": True, "evaluate_test": False},
]
mlp_results = run_parallel_trainings(MLP_SPECS, max_workers=2)

Loaded cached model: default_90d5d0eaeb826ab5.pt | val R²=0.737 RMSE=0.673 MAE=0.486
Loaded cached model: default_de0b844eb81e39cd.pt | val R²=0.558 RMSE=0.784 MAE=0.580


In [6]:
SUMMARY_COLS = ["model", "split", "seed", "lr", "batch_size", "weight_decay",
                "avg_train_loss", "avg_val_loss", "best_val_loss", "r2_val", "rmse_val", "mae_val", "from_cache"]

if results_table:
    df_mlp = pl.DataFrame(results_table).filter(pl.col("model") == "MLP")
    cols = [c for c in SUMMARY_COLS if c in df_mlp.columns]
    display(df_mlp.select(cols).sort("r2_val", descending=True))
else:
    print("Brak wynikow. Uruchom najpierw komorke treningowa MLP.")

model,split,seed,lr,batch_size,weight_decay,avg_train_loss,avg_val_loss,best_val_loss,r2_val,rmse_val,mae_val,from_cache
str,str,i64,f64,i64,f64,f64,f64,f64,f64,f64,f64,bool
"""MLP""","""random""",42,0.0003,64,0.00001,0.181246,0.217912,0.200916,0.737153,0.67308,0.486056,true
"""MLP""","""scaffold""",42,0.0003,64,0.00001,0.187594,0.316156,0.289175,0.558177,0.783503,0.57975,true


In [7]:
BEST_GNN_CONFIGS = {
    "scaffold": {"lr": 3e-4, "weight_decay": 1e-5, "pooling": "mean",
                 "gnn_hidden_dim": 192, "gnn_num_layers": 4, "gnn_dropout": 0.10},
    "random":   {"lr": 3e-4, "weight_decay": 1e-5, "pooling": "mean",
                 "gnn_hidden_dim": 192, "gnn_num_layers": 4, "gnn_dropout": 0.10},
}

GNN_SPECS = []
for split_type, cfg in BEST_GNN_CONFIGS.items():
    spec = {"model_type": "GNN", "split_type": split_type, "df_fp": df_clean,
            "epochs": 100, "batch_size": 64, "seed": 42,
            "log_mlflow": True, "evaluate_test": False,
            "early_stopping_patience": 12, "min_delta": 1e-4,
            "prefer_cuda": True, **cfg}
    GNN_SPECS.append(spec)

gnn_results = run_parallel_trainings(GNN_SPECS, max_workers=2)

Loaded cached model: default_718ff50075a1ad9d.pt | val R²=0.678 RMSE=0.745 MAE=0.550
Loaded cached model: default_791c4e5308e73d02.pt | val R²=0.468 RMSE=0.843 MAE=0.632


In [8]:
PREFERRED_COLS = [
    "model", "split", "seed", "epochs", "epochs_trained", "best_epoch",
    "lr", "batch_size", "weight_decay", "pooling",
    "gnn_hidden_dim", "gnn_num_layers", "gnn_dropout",
    "avg_train_loss", "avg_val_loss", "best_val_loss", "r2_val", "rmse_val", "mae_val", "from_cache",
]

if results_table:
    df_all = pl.DataFrame(results_table)
    cols = [c for c in PREFERRED_COLS if c in df_all.columns]
    display(df_all.select(cols).sort("r2_val", descending=True))
else:
    print("Brak wynikow. Uruchom najpierw komorki treningowe.")

model,split,seed,epochs,epochs_trained,best_epoch,lr,batch_size,weight_decay,pooling,gnn_hidden_dim,gnn_num_layers,gnn_dropout,avg_train_loss,avg_val_loss,best_val_loss,r2_val,rmse_val,mae_val,from_cache
str,str,i64,i64,i64,i64,f64,i64,f64,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,bool
"""MLP""","""random""",42,200,73,58,0.0003,64,0.00001,"""mean""",128,4,0.15,0.181246,0.217912,0.200916,0.737153,0.67308,0.486056,true
"""GNN""","""random""",42,100,100,100,0.0003,64,0.00001,"""mean""",192,4,0.1,0.339977,0.346325,0.240884,0.677887,0.745107,0.549885,true
"""MLP""","""scaffold""",42,200,60,45,0.0003,64,0.00001,"""mean""",128,4,0.15,0.187594,0.316156,0.289175,0.558177,0.783503,0.57975,true
"""GNN""","""scaffold""",42,100,86,74,0.0003,64,0.00001,"""mean""",192,4,0.1,0.350199,0.360227,0.302081,0.467704,0.84263,0.631512,true


In [9]:
# Finalny test: odkomentuj po wybraniu najlepszego wariantu.
best = pl.DataFrame(results_table).sort("r2_val", descending=True).row(0, named=True)
final_result = train_and_score(
    model_type=best["model"],
    split_type=best["split"],
    df_fp=df_clean,
    epochs=200,
    seed=42,
    log_mlflow=True,
    evaluate_test=True,
    prefer_cuda=True,
)

Loaded cached model: default_90d5d0eaeb826ab5.pt | val R²=0.737 RMSE=0.673 MAE=0.486


In [10]:
# --- ETAP 1: coarse grid ---
# fix: hidden_dim=192, layers=4, weight_decay=1e-5
# grid: lr x pooling x dropout = 2 x 3 x 3 = 18 configs x 2 seeds = 36 runs

stage1_search = {
    "lr": [3e-4, 5e-4],
    "weight_decay": [1e-5],
    "pooling": ["mean", "add", "attention"],
    "gnn_hidden_dim": [192],
    "gnn_num_layers": [4],
    "gnn_dropout": [0.05, 0.1, 0.15],
}

print(f"=== ETAP 1: {2*3*3} configow x 2 seedy = {2*3*3*2} treningow ===")

stage1 = tune_gnn(
    split_type="scaffold",
    search_space=stage1_search,
    seeds=[42, 123],
    df_fp=df_clean,
    epochs=200,
    early_stopping_patience=15,
    log_mlflow=True,
    prefer_cuda=True,
    max_workers=2,
)

print("\nTop 5 coarse grid:")
print(stage1.head(5))

# --- ETAP 2: fine tuning wokol zwyciezcy ---
best = stage1.row(0, named=True)
print(f"\n=== ETAP 2: fine tuning wokol {best['pooling']}, dropout={best['gnn_dropout']}, lr={best['lr']} ===")

stage2_search = {
    "lr": [max(1e-4, best['lr']/2), best['lr'], min(1e-3, best['lr']*2)],
    "weight_decay": [1e-5],
    "pooling": [best['pooling']],
    "gnn_hidden_dim": [128, 192, 256],
    "gnn_num_layers": [4, 5],
    "gnn_dropout": [max(0.0, best['gnn_dropout']-0.05), best['gnn_dropout'], min(0.3, best['gnn_dropout']+0.05)],
}

stage2 = tune_gnn(
    split_type="scaffold",
    search_space=stage2_search,
    seeds=[42, 123],
    df_fp=df_clean,
    epochs=200,
    early_stopping_patience=15,
    log_mlflow=True,
    prefer_cuda=True,
    max_workers=2,
)

print("\nTop 5 fine-tuned:")
print(stage2.head(5))

=== ETAP 1: 18 configow x 2 seedy = 36 treningow ===
Loaded cached model: gnn_tuning_82875c0c234e0f45.pt | val R²=0.355 RMSE=0.862 MAE=0.652
Loaded cached model: gnn_tuning_73403a752c3cecbd.pt | val R²=0.400 RMSE=0.872 MAE=0.654
Loaded cached model: gnn_tuning_a6f4f4cc3df261f3.pt | val R²=0.476 RMSE=0.807 MAE=0.605
[scaffold] config 2/18 done | R² mean=0.415 ± 0.060
Loaded cached model: gnn_tuning_08b2658779058465.pt | val R²=0.433 RMSE=0.880 MAE=0.642
[scaffold] config 1/18 done | R² mean=0.417 ± 0.017
Loaded cached model: gnn_tuning_7e9658ab2215dd0e.pt | val R²=0.333 RMSE=0.914 MAE=0.671
Loaded cached model: gnn_tuning_89cb1f14d23c0926.pt | val R²=0.333 RMSE=0.922 MAE=0.692
Loaded cached model: gnn_tuning_9532f88ef72c5fb4.pt | val R²=0.361 RMSE=0.921 MAE=0.673
[scaffold] config 4/18 done | R² mean=0.347 ± 0.014
GNN | scaffold | device=cuda: avg train loss=0.4686, avg val loss=0.4295, best val loss=0.3462, best epoch=62, val R²=0.352 RMSE=0.941 MAE=0.687
GNN | scaffold | device=cuda: 

# Wnioski

- Ranking wariantow na podstawie **R² walidacyjnego** (im wyzszy, tym lepszy), pomocniczo **RMSE** (im nizszy, tym lepszy).
- Dla GNN raportuj srednia i odchylenie standardowe po wielu seedach (minimum 3).
- Scaffold split jako glowny test uogolniania chemicznego na nowe rusztowania.
- Family split dla testu uogolniania miedzy rodzinami zwiazkow chemicznych.
- Najpierw wybierz najlepsza konfiguracje po `gnn_scaffold_tuning`, dopiero potem finalny test.
- Model przewiduje pIC50 bezposrednio (regresja); RMSE < 1.0 oznacza blad ponizej rzedu wielkosci.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

if results_table:
    df_all = pl.DataFrame(results_table)
    df_best = (
        df_all
        .group_by("model", "split")
        .agg(pl.all().sort_by("r2_val", descending=True).first())
        .sort("r2_val", descending=True)
    )

    labels = [f"{r['model']}-{r['split']}" for r in df_best.iter_rows(named=True)]
    x = np.arange(len(labels))
    width = 0.25

    r2 = df_best["r2_val"].to_list()
    rmse = df_best["rmse_val"].to_list()
    mae = df_best["mae_val"].to_list()

    fig, ax = plt.subplots(figsize=(10, 4))
    fig.suptitle(
        "Porównanie najlepszych modeli GNN i MLP",
        fontsize=14, fontweight="bold", y=1.02,
    )

    bars1 = ax.bar(x - width, r2, width, label="R²",
                   color="#2ecc71", edgecolor="white")
    bars2 = ax.bar(x, rmse, width, label="RMSE",
                   color="#e74c3c", edgecolor="white")
    bars3 = ax.bar(x + width, mae, width, label="MAE",
                   color="#f39c12", edgecolor="white")

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel("Wartość", fontsize=11)
    ax.legend(fontsize=10)
    ax.axhline(y=0, color="gray", linewidth=0.8)
    ax.grid(axis="y", alpha=0.3)

    for bar, color in zip(bars1, ["#2ecc71"] * len(bars1)):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha="center", va="bottom",
                fontsize=8, color=color, fontweight="bold")
    for bar, color in zip(bars2, ["#e74c3c"] * len(bars2)):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha="center", va="bottom",
                fontsize=8, color=color, fontweight="bold")
    for bar, color in zip(bars3, ["#f39c12"] * len(bars3)):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha="center", va="bottom",
                fontsize=8, color=color, fontweight="bold")

    plt.tight_layout()
    plt.show()

    COLS_SHOW = ["model", "split", "r2_val", "rmse_val",
                 "mae_val", "best_val_loss", "best_epoch",
                 "epochs_trained", "lr", "batch_size"]
    display(df_best.select(COLS_SHOW))
else:
    print("Brak wyników. Uruchom najpierw komórki treningowe.")

NameError: name 'results_table' is not defined